# Plug-in solar at 10% household adoption

Estimates bill savings from small plug-in solar systems (391W and 1,200W) across all U.S. households:

1. **Per adopting household:** output, share used, and annual bill savings, by home type.
2. **10% adoption:** national annual and 10-year bill savings if 10% of households adopt.
3. **By state:** the same totals for each state.

**Methodology**
1. Sample 5,000 occupied housing units from ResStock (all building types, tenures and heating fuels).
2. Pull each home's hourly electricity use (no upgrades).
3. Simulate plug-in solar with PVWatts: houses and mobile homes get a south-facing array at 20°; apartments get a vertical railing array on the back of the unit.
4. Each hour the home uses what it can of the output; surplus is lost (no export compensation). Savings are the used kWh at the state's marginal electricity rate.
5. Scale to 10% of each state's households, drawn from homes that would adopt (excluding apartments with north-facing balconies), and sum to a national total. 10-year savings apply 0.5%/yr panel degradation at today's rates.

**Scope.** Occupied housing units in the lower 48 states and DC (ResStock does not cover Alaska and Hawaii). System costs, shading, and changes in electricity rates are not included.

**Sections.** 1. Sample · 2. Mounting and adoption · 3. Hourly simulation · 4. Electricity rates · Results 1–3

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../src"))

from solar_bill_analysis import (
    ORIENTATION_TO_AZIMUTH,
    build_building_hourly_profiles,
    compute_plugin_solar,
    load_resstock_metadata,
    orientation_to_azimuth,
)
from eia_rates import build_state_residential_rates

## 1. Sample

ResStock 2024 Release 2 baseline (TMY3 weather), occupied units only. Every unit carries the same sample weight, so the sample is nationally representative and the weights give household counts.

In [2]:
SAMPLE_SIZE = 5000
RANDOM_SEED = 42

metadata = load_resstock_metadata(upgrade="0", filters={"in.vacancy_status": "Occupied"})
assert metadata["weight"].nunique() == 1

households_by_state = metadata.groupby("in.state")["weight"].sum()
sample = metadata.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED)

print(f"Occupied households represented: {households_by_state.sum() / 1e6:,.1f}M across {len(households_by_state)} states")
print(f"Sampled homes: {len(sample):,}")
print((sample["in.geometry_building_type_recs"].value_counts(normalize=True) * 100).round(1).to_string())

Occupied households represented: 121.9M across 49 states
Sampled homes: 5,000
in.geometry_building_type_recs
Single-Family Detached           62.5
Multi-Family with 5+ Units       18.4
Multi-Family with 2 - 4 Units     7.8
Single-Family Attached            5.8
Mobile Home                       5.4


## 2. Mounting and adoption

- **Houses and mobile homes** (single-family detached and attached, mobile homes): yard, patio or roof-edge mount facing south at a 20° tilt.
- **Apartments** (2–4 and 5+ units): vertical (90°) railing mount on the balcony. ResStock assigns each unit's front a direction drawn from a national distribution (`in.orientation`, "orientation of the front of the housing unit as it faces the street") and models apartments with an interior corridor at the front, so the balcony is placed on the back: `in.orientation` + 180°.
- **Adoption:** a random 10% of each state's households, drawn from homes that would adopt. ASSUMPTION: apartments whose balcony faces north, northeast or northwest don't adopt.
- **10-year savings:** 0.5%/yr panel degradation, with the share of output used held constant; today's electricity rates.

In [3]:
PLUGIN_SIZES_KW = (0.391, 1.2)
ADOPTION_RATE = 0.10
YEARS = 10
DEGRADATION_RATE = 0.005

HOUSE_TYPES = ("Single-Family Detached", "Single-Family Attached", "Mobile Home")
APARTMENT_TYPES = ("Multi-Family with 2 - 4 Units", "Multi-Family with 5+ Units")
HOUSE_TILT_DEG, HOUSE_AZIMUTH_DEG = 20.0, 180.0
RAILING_TILT_DEG = 90.0
NON_ADOPTING_BALCONY_DIRECTIONS = ("North", "Northeast", "Northwest")

building_type = sample["in.geometry_building_type_recs"]
assert building_type.isin(HOUSE_TYPES + APARTMENT_TYPES).all()
is_apartment = building_type.isin(APARTMENT_TYPES)

balcony_azimuth = (sample["in.orientation"].map(orientation_to_azimuth) + 180.0) % 360.0
direction_by_azimuth = {az: name for name, az in ORIENTATION_TO_AZIMUTH.items()}

mounting = pd.DataFrame({
    "bldg_id": sample["bldg_id"],
    "home_type": np.where(is_apartment, "Apartment", "House or mobile home"),
    "tilt_deg": np.where(is_apartment, RAILING_TILT_DEG, HOUSE_TILT_DEG),
    "azimuth_deg": np.where(is_apartment, balcony_azimuth, HOUSE_AZIMUTH_DEG),
    "panel_direction": np.where(is_apartment, balcony_azimuth.map(direction_by_azimuth), "South"),
})
mounting["adopts"] = ~(is_apartment.to_numpy() & mounting["panel_direction"].isin(NON_ADOPTING_BALCONY_DIRECTIONS))

print(mounting.groupby("home_type")["adopts"].agg(["size", "mean"]).rename(columns={"size": "homes", "mean": "share that would adopt"}))

                      homes  share that would adopt
home_type                                          
Apartment              1313                0.664128
House or mobile home   3687                1.000000


## 3. Hourly simulation

For each batch of 500 homes: pull hourly electricity use from S3 and simulate both plug-in sizes with each home's mounting. Only annual totals are kept, and each batch is cached to `../data/` so re-runs skip the downloads and PVWatts runs.

In [4]:
BUCKET = "oedi-data-lake"
TIMESERIES_BASE_PREFIX = (
    "nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/"
    "2024/resstock_tmy3_release_2/timeseries_individual_buildings/by_state"
)
WEATHER_BASE_PREFIX = (
    "https://oedi-data-lake.s3.amazonaws.com/"
    "nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/"
    "2024/resstock_tmy3_release_2/weather"
)
BATCH_SIZE = 500
CACHE_DIR = Path(f"../data/plugin_penetration_all_households_{SAMPLE_SIZE}_seed{RANDOM_SEED}")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

n_batches = -(-len(sample) // BATCH_SIZE)
batches = []
for i, start in enumerate(range(0, len(sample), BATCH_SIZE), start=1):
    batch_path = CACHE_DIR / f"batch_{i:02d}.csv"
    if batch_path.exists():
        batches.append(pd.read_csv(batch_path, dtype={"bldg_id": str}))
        continue

    print(f"Batch {i} of {n_batches}")
    meta_batch = sample.iloc[start:start + BATCH_SIZE]

    hourly_batch = build_building_hourly_profiles(
        bldg_ids=meta_batch["bldg_id"].tolist(),
        metadata_df=meta_batch,
        upgrades=("0",),
        bucket=BUCKET,
        base_prefix=TIMESERIES_BASE_PREFIX,
        region="us-west-2",
        chunk_size=250,
        max_workers=8,
    )
    batch = compute_plugin_solar(
        hourly_batch, meta_batch, WEATHER_BASE_PREFIX,
        sizes_kw=PLUGIN_SIZES_KW,
        orientations=mounting[mounting["bldg_id"].isin(meta_batch["bldg_id"])],
        upgrade="0",
    )
    batch.to_csv(batch_path, index=False)
    batches.append(batch)

annual = pd.concat(batches, ignore_index=True)
assert len(annual) == len(sample)
print(f"Homes simulated: {len(annual):,}")

Batch 1 of 10


Batch 2 of 10


Batch 3 of 10


Batch 4 of 10


Batch 5 of 10


...

Batch 7 of 10


Batch 8 of 10


Batch 9 of 10


Batch 10 of 10


Homes simulated: 5,000


## 4. Electricity rates

EIA state residential rates over the latest 12 months, with a $10/month fixed customer charge removed. Plug-in solar doesn't change fixed charges, so savings use the marginal rate.

In [5]:
eia_rates = build_state_residential_rates()

homes = (
    annual
    .merge(mounting, on="bldg_id")
    .merge(sample[["bldg_id", "in.state"]], on="bldg_id")
    .merge(eia_rates[["state", "elec_marginal_cents_per_kwh"]], left_on="in.state", right_on="state")
)
assert len(homes) == len(sample)
homes["elec_rate"] = homes["elec_marginal_cents_per_kwh"] / 100  # $/kWh

plugin_sizes = {f"plugin_{round(kw * 1000)}w": f"{round(kw * 1000):,}W" for kw in PLUGIN_SIZES_KW}
for col in plugin_sizes:
    assert (homes[f"{col}_used_kwh"] <= homes[f"{col}_production_kwh"] + 1e-9).all()
    homes[f"{col}_savings"] = homes[f"{col}_used_kwh"] * homes["elec_rate"]

# Sum of yearly output over the period relative to year 1, with 0.5%/yr degradation
ten_year_factor = sum((1 - DEGRADATION_RATE) ** t for t in range(YEARS))

print(f"EIA rates: {eia_rates['period_start'].iloc[0]} to {eia_rates['period_end'].iloc[0]}")
print(f"{YEARS}-year savings = annual savings × {ten_year_factor:.3f}")

EIA rates: 2025-08 to 2026-07
10-year savings = annual savings × 9.778


## Result 1. Plug-in solar per adopting household

Means and medians across sampled homes that would adopt, by home type.

In [6]:
adopters = homes[homes["adopts"]]

def per_household(df: pd.DataFrame) -> pd.Series:
    row = {"Homes": len(df)}
    for col, size in plugin_sizes.items():
        row[f"{size}: production (mean kWh/yr)"] = df[f"{col}_production_kwh"].mean()
        row[f"{size}: share used"] = df[f"{col}_used_kwh"].sum() / df[f"{col}_production_kwh"].sum()
        row[f"{size}: bill savings (mean $/yr)"] = df[f"{col}_savings"].mean()
        row[f"{size}: bill savings (median $/yr)"] = df[f"{col}_savings"].median()
    return pd.Series(row)

result1 = pd.concat(
    [per_household(adopters).rename("All adopting households")]
    + [per_household(g).rename(t) for t, g in adopters.groupby("home_type")],
    axis=1,
).T
result1.index.name = "Home type"

print(result1.T.to_string(float_format=lambda v: f"{v:,.2f}"))
result1.to_csv("../outputs/plugin_solar_per_household.csv")

Home type                           All adopting households  Apartment  House or mobile home
Homes                                              4,559.00     872.00              3,687.00
391W: production (mean kWh/yr)                       473.70     293.54                516.31
391W: share used                                       1.00       0.99                  1.00
391W: bill savings (mean $/yr)                        86.11      58.97                 92.53
391W: bill savings (median $/yr)                      78.13      50.08                 80.11
1,200W: production (mean kWh/yr)                   1,453.81     900.88              1,584.58
1,200W: share used                                     0.92       0.89                  0.92
1,200W: bill savings (mean $/yr)                     240.90     159.87                260.07
1,200W: bill savings (median $/yr)                   226.18     139.71                234.05


## Result 2. National savings at 10% adoption

Adopting households in each state = 10% of the state's households; each saves the state's mean savings among sampled homes that would adopt. State totals are summed to the national total.

In [7]:
adoption_share_by_state = homes.groupby("in.state")["adopts"].mean()
assert (adoption_share_by_state >= ADOPTION_RATE).all(), "fewer eligible homes than the adoption rate in some state"

state_rows = []
for state, g in adopters.groupby("in.state"):
    adopting_households = ADOPTION_RATE * households_by_state[state]
    row = {"State": state, "Sampled homes that would adopt": len(g), "Adopting households": adopting_households}
    for col, size in plugin_sizes.items():
        annual_savings = adopting_households * g[f"{col}_savings"].mean()
        row[f"{size}: annual savings ($M)"] = annual_savings / 1e6
        row[f"{size}: {YEARS}-year savings ($M)"] = annual_savings * ten_year_factor / 1e6
        row[f"{size}: annual energy offset (GWh)"] = adopting_households * g[f"{col}_used_kwh"].mean() / 1e6
    state_rows.append(row)
result3 = pd.DataFrame(state_rows).set_index("State")
assert set(result3.index) == set(households_by_state.index)

result2 = result3.drop(columns="Sampled homes that would adopt").sum().rename("United States (lower 48 + DC)")
print(result2.to_string(float_format=lambda v: f"{v:,.1f}"))
result2.to_frame().T.to_csv("../outputs/plugin_solar_10pct_adoption_national.csv", index_label="Area")

Adopting households                  12,194,797.9
391W: annual savings ($M)                 1,061.4
391W: 10-year savings ($M)               10,378.3
391W: annual energy offset (GWh)          5,771.4
1,200W: annual savings ($M)               2,963.4
1,200W: 10-year savings ($M)             28,976.5
1,200W: annual energy offset (GWh)       16,297.0


## Result 3. Savings at 10% adoption by state

States with fewer than about 30 sampled homes are noisy.

In [8]:
print(result3.to_string(float_format=lambda v: f"{v:,.1f}"))
result3.to_csv("../outputs/plugin_solar_10pct_adoption_by_state.csv")

       Sampled homes that would adopt  Adopting households  391W: annual savings ($M)  391W: 10-year savings ($M)  391W: annual energy offset (GWh)  1,200W: annual savings ($M)  1,200W: 10-year savings ($M)  1,200W: annual energy offset (GWh)
State                                                                                                                                                                                                                                             
AL                                 59            190,815.7                       14.4                       140.5                              92.4                         42.7                         417.6                               274.6
AR                                 41            117,925.8                        7.5                        73.1                              60.0                         21.5                         210.0                               172.2
AZ                          